# 03 — Traffic Calibration & Time-Dependent Routing

Notebook này nối tiếp `02_Build_OSMnx_Network.ipynb`.

## Mục tiêu
- dùng `incident_hospital_od.csv.gz` và `hospital_charger_od.csv` làm **network free-flow baseline**;
- tạo lớp `traffic_multiplier = γ(t,ω)`;
- không gọi free-flow OSMnx là traffic thực tế;
- xuất bảng travel-time theo scenario/replication dùng cho DES.

## Quan trọng
Report chỉ hỗ trợ công thức:

\[
T_{uv}(t)=T^{freeflow}_{uv}	imes \gamma_{uv}(t,\omega)
\]

nhưng **không cung cấp calibration local đủ mạnh cho TP.HCM**. Vì vậy notebook mặc định chạy chế độ `freeflow_reference` với `γ=1.0`. Khi có nguồn traffic hợp lệ, thay profile trong config và ghi provenance.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, cwd.parent, *cwd.parents] if (p/"data"/"sample").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy project root.")

OSM = ROOT/"data"/"osmnx"
DATA = ROOT/"data"/"sample"
OUT = ROOT/"data"/"routing"
OUT.mkdir(parents=True, exist_ok=True)

inc = pd.read_csv(DATA/"incidents.csv.gz")
reps = pd.read_csv(DATA/"replications.csv.gz")

IH = OSM/"incident_hospital_od.csv.gz"
HC = OSM/"hospital_charger_od.csv"

print("ROOT:", ROOT)
print("Incident-Hospital OD exists:", IH.exists())
print("Hospital-Charger OD exists:", HC.exists())

## 1. Traffic profile contract

Để không bịa calibration, profile mặc định là **identity**:

```text
gamma = 1.0
```

Khi thay bằng dữ liệu thật/synthetic calibrated, phải lưu:
- `profile_id`
- `source_type`
- `source_name`
- `retrieved_at`
- `license_or_terms`
- `calibration_status`

In [ ]:
TRAFFIC_PROFILE = {
    "profile_id": "freeflow_reference_v1",
    "source_type": "assumption",
    "source_name": "OSMnx free-flow baseline only",
    "calibration_status": "NOT_CALIBRATED_TO_HCMC_TRAFFIC",
    "gamma_default": 1.0,
}

(ROOT/"configs").mkdir(exist_ok=True)
(ROOT/"configs"/"traffic_profile.json").write_text(
    json.dumps(TRAFFIC_PROFILE, indent=2),
    encoding="utf-8"
)
TRAFFIC_PROFILE

## 2. Incident-level time multiplier

In [ ]:
inc["timestamp"] = pd.to_datetime(inc["timestamp"], utc=True)
inc["traffic_multiplier"] = TRAFFIC_PROFILE["gamma_default"]
inc[["scenario_id","replication_id","incident_id","timestamp","traffic_multiplier"]].head()

## 3. Join free-flow OD → time-dependent travel

In [ ]:
if not IH.exists():
    print("Chưa có OSMnx OD. Hãy chạy Notebook 02 với RUN_NETWORK=True trước.")
    incident_hospital_time = None
else:
    od = pd.read_csv(IH)
    incident_hospital_time = od.merge(
        inc[["scenario_id","replication_id","incident_id","timestamp","traffic_multiplier"]],
        on=["scenario_id","replication_id","incident_id"],
        how="left",
        validate="many_to_one",
    )
    incident_hospital_time["travel_time_s"] = (
        incident_hospital_time["freeflow_shortest_travel_time_s"]
        * incident_hospital_time["traffic_multiplier"]
    )
    out = OUT/"incident_hospital_time_dependent.csv.gz"
    incident_hospital_time.to_csv(out, index=False, compression="gzip")
    print("Saved:", out, "rows:", len(incident_hospital_time))

## 4. QA

In [ ]:
if incident_hospital_time is not None:
    display(incident_hospital_time[["freeflow_shortest_travel_time_s","traffic_multiplier","travel_time_s"]].describe())
    fig, ax = plt.subplots(figsize=(8,4.5))
    ax.hist(incident_hospital_time.loc[incident_hospital_time["reachable"], "travel_time_s"]/60, bins=40)
    ax.set_xlabel("Travel time (min)")
    ax.set_ylabel("OD count")
    ax.set_title("Incident → Hospital travel time")
    ax.grid(axis="y", alpha=.25)
    plt.show()

## Gate để sang Notebook 04
- OSM graph đã lưu;
- snap QA hợp lý;
- Incident→Hospital OD có reachability hợp lý;
- traffic profile có provenance;
- nếu `γ=1.0`, kết quả chỉ là **free-flow reference**, chưa phải HCMC traffic experiment.